In [1]:
import requests as rq
import pandas as pd
import datetime as dt
import json

In [2]:
with open('accessToken.txt',"r") as file:
    access_token = file.read()
url = "https://api.upstox.com/v2/user/profile"
headers = {
    'accept': 'application/json',
    'Api-Version': '2.0',
    'Authorization': f'Bearer {access_token}'}
response = rq.get(url, headers=headers)
print(response.json()['status'])

success


In [10]:
df = pd.read_csv("https://assets.upstox.com/market-quote/instruments/exchange/complete.csv.gz")

In [11]:
def filter_df(df,lot_size):
    df = df[(df['exchange'] == 'NSE_FO') &(df['instrument_type'] == 'OPTIDX') &(df['lot_size'] == lot_size)]
    df = df[df['expiry'] ==  min(df['expiry'].unique())]
    return df

In [12]:
BNDF = filter_df(df,15)
BNDF[(BNDF['strike'] == 48100) & (BNDF['option_type'] == 'PE')]['instrument_key'].iloc[0]

'NSE_FO|50471'

In [5]:

url = "https://api.upstox.com/v2/order/place"
headers = {
    'accept': 'application/json',
    'Api-Version': '2.0',
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {access_token}'}
payload={
  "quantity": 15,
  "product": "D",
  "validity": "DAY",
  "price": 0,
  "tag": "string",
  "instrument_token": "NSE_FO|41384",
  "order_type": "MARKET",
  "transaction_type": "BUY",
  "disclosed_quantity": 0,
  "trigger_price": 0,
  "is_amo": False}
data = json.dumps(payload)

response = rq.post(url, headers=headers, data=data)

print(response.json())

{'status': 'success', 'data': {'order_id': '240316010003512'}}


In [40]:
oid = response.json()['data']['order_id']
oid

'240104010854534'

In [41]:
url = "https://api.upstox.com/v2/order/cancel"
headers = {
    'accept': 'application/json',
    'Api-Version': '2.0',
    'Authorization': f'Bearer {access_token}'}
payload={"order_id":oid}
res = rq.delete(url, headers=headers, params=payload)
print(res.json())

{'status': 'success', 'data': {'order_id': '240104010854534'}}


In [17]:
def get_order_history(oid):
    url = "https://api.upstox.com/v2/order/details"
    payload={'order_id':oid}
    headers = {
        'accept': 'application/json',
        'Api-Version': '2.0',
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {access_token}'}
    response = rq.get( url, headers=headers,params=payload)
    return response.json()['data']
od= get_order_history('240316010003512')
od

{'exchange': 'NFO',
 'product': 'D',
 'price': 0.0,
 'quantity': 15,
 'status': 'rejected',
 'guid': None,
 'tag': 'string',
 'instrument_token': 'NSE_FO|41384',
 'placed_by': '7NBGNG',
 'tradingsymbol': 'BANKNIFTY2432046900CE',
 'trading_symbol': 'BANKNIFTY2432046900CE',
 'order_type': 'MARKET',
 'validity': 'DAY',
 'trigger_price': 0.0,
 'disclosed_quantity': 0,
 'transaction_type': 'BUY',
 'average_price': 0.0,
 'filled_quantity': 0,
 'pending_quantity': 0,
 'status_message': "Markets are closed right now, you can't place this order. Place an After-Market-Order (AMO) instead. Select 'AMO' under 'Order Complexity' from the order entry screen.",
 'status_message_raw': 'Adapter is Logged Off',
 'exchange_order_id': '',
 'parent_order_id': None,
 'order_id': '240316010003512',
 'variety': 'SIMPLE',
 'order_timestamp': '2024-03-16 16:11:51',
 'exchange_timestamp': None,
 'is_amo': False,
 'order_request_id': '1',
 'order_ref_id': 'UDAPI-I-Dq7ZfzIrymC9rFGT5nqkByh8'}

In [8]:
oid = '240316010003512'
for i in od:
    if i['order_id'] == oid and i['status'] == 'complete':
        print(i)
        avg_prc = i['average_price']
        i['status']
        break

{'exchange': 'NFO', 'product': 'D', 'price': 0.0, 'quantity': 15, 'status': 'rejected', 'guid': None, 'tag': 'string', 'instrument_token': 'NSE_FO|41384', 'placed_by': '7NBGNG', 'tradingsymbol': 'BANKNIFTY2432046900CE', 'trading_symbol': 'BANKNIFTY2432046900CE', 'order_type': 'MARKET', 'validity': 'DAY', 'trigger_price': 0.0, 'disclosed_quantity': 0, 'transaction_type': 'BUY', 'average_price': 0.0, 'filled_quantity': 0, 'pending_quantity': 0, 'status_message': "Markets are closed right now, you can't place this order. Place an After-Market-Order (AMO) instead. Select 'AMO' under 'Order Complexity' from the order entry screen.", 'status_message_raw': 'Adapter is Logged Off', 'exchange_order_id': '', 'parent_order_id': None, 'order_id': '240316010003512', 'variety': 'SIMPLE', 'order_timestamp': '2024-03-16 16:11:51', 'exchange_timestamp': None, 'is_amo': False, 'order_request_id': '1', 'order_ref_id': 'UDAPI-I-Dq7ZfzIrymC9rFGT5nqkByh8'}


In [46]:
import pandas as pd

In [47]:
pd.DataFrame(response.json()['data'])

,exchange,product,price,quantity,status,guid,tag,instrument_token,placed_by,tradingsymbol,...,status_message_raw,exchange_order_id,parent_order_id,order_id,variety,order_timestamp,exchange_timestamp,is_amo,order_request_id,order_ref_id
0,NFO,I,0.0,15,rejected,None,tradetool,NSE_FO|50472,7NBGNG,BANKNIFTY2411048200CE,...,Adapter is Logged Off,,None,240104010853444,SIMPLE,2024-01-04 17:52:01,None,False,1,UDAPI-I-EQkW2mYJ0FeQ86Anuj3b1r3s
1,NFO,I,0.0,15,rejected,None,tradetool,NSE_FO|50472,7NBGNG,BANKNIFTY2411048200CE,...,Adapter is Logged Off,,None,240104010853478,SIMPLE,2024-01-04 17:52:58,None,False,1,UDAPI-I-et1fh3uozcQeIONXVILXvobX
2,NFO,I,0.0,15,rejected,None,tradetool,NSE_FO|50472,7NBGNG,BANKNIFTY2411048200CE,...,Adapter is Logged Off,,None,240104010853545,SIMPLE,2024-01-04 17:54:15,None,False,1,UDAPI-I-plshAOiW5VEnmCVYeyJsOmvy
3,NFO,I,0.0,15,rejected,None,tradetool,NSE_FO|50472,7NBGNG,BANKNIFTY2411048200CE,...,Adapter is Logged Off,,None,240104010853563,SIMPLE,2024-01-04 17:54:38,None,False,1,UDAPI-I-vCLfL51hYvRNDOBz50eNsaOq
4,NFO,D,0.0,15,rejected,None,string,NSE_FO|50471,7NBGNG,BANKNIFTY2411048100PE,...,Adapter is Logged Off,,None,240104010854247,SIMPLE,2024-01-04 18:11:24,None,False,1,UDAPI-I-D0DAkir8yGvDFHQ41iNRJpH4
5,NFO,D,0.0,15,rejected,None,string,NSE_FO|50471,7NBGNG,BANKNIFTY2411048100PE,...,Adapter is Logged Off,,None,240104010854270,SIMPLE,2024-01-04 18:12:10,None,False,1,UDAPI-I-uSuurMz2jaAmEKiO06uVxdeT
6,NFO,D,0.0,15,rejected,None,string,NSE_FO|50471,7NBGNG,BANKNIFTY2411048100PE,...,Adapter is Logged Off,,None,240104010854294,SIMPLE,2024-01-04 18:12:50,None,False,1,UDAPI-I-4zZVanG8JT38QI78XzTYtO39
7,NFO,D,150.0,15,cancelled after market order,None,string,NSE_FO|50471,7NBGNG,BANKNIFTY2411048100PE,...,None,,None,240104010854303,SIMPLE,2024-01-04 18:18:44,None,False,1,5733784998879069694-7NBGNG-JAPIA
8,NFO,D,150.0,15,cancelled after market order,None,string,NSE_FO|50471,7NBGNG,BANKNIFTY2411048100PE,...,None,,None,240104010854496,SIMPLE,2024-01-04 18:19:06,None,False,1,6145412932440449182-7NBGNG-JAPIA
9,NFO,D,150.0,15,cancelled after market order,None,string,NSE_FO|50471,7NBGNG,BANKNIFTY2411048100PE,...,None,,None,240104010854534,SIMPLE,2024-01-04 18:20:15,None,False,1,7023575124240948043-7NBGNG-JAPIA


In [50]:
# url = "https://api.coindcx.com/exchange/v1/markets"
url = "https://api.coindcx.com/exchange/v1/markets_details"
res = rq.get(url)
data = res.json()
print(data)


[{'coindcx_name': 'EDUBTC', 'base_currency_short_name': 'BTC', 'target_currency_short_name': 'EDU', 'target_currency_name': 'Open Campus', 'base_currency_name': 'Bitcoin', 'min_quantity': 1.0, 'max_quantity': 92141578.0, 'max_quantity_market': 10000000.0, 'min_price': 1e-08, 'max_price': 1000.0, 'min_notional': 0.001, 'base_currency_precision': 8, 'target_currency_precision': 0, 'step': 1.0, 'order_types': ['take_profit', 'stop_limit', 'market_order', 'limit_order'], 'symbol': 'EDUBTC', 'ecode': 'B', 'bo_sl_safety_percent': None, 'max_leverage': None, 'max_leverage_short': None, 'pair': 'B-EDU_BTC', 'status': 'active'}, {'coindcx_name': 'CYBERETH', 'base_currency_short_name': 'ETH', 'target_currency_short_name': 'CYBER', 'target_currency_name': 'CyberConnect', 'base_currency_name': 'Ethereum', 'min_quantity': 0.01, 'max_quantity': 92141578.0, 'max_quantity_market': 10000000.0, 'min_price': 1e-06, 'max_price': 1000.0, 'min_notional': 0.01, 'base_currency_precision': 6, 'target_currency_

In [60]:
inr = []
for i in data:
    if i['status'] == 'active' and (i['base_currency_short_name'] == 'INR' or i['base_currency_short_name'] == 'USDT') :
        if i['pair'].startswith("I") or i['pair'].startswith("B"):
            inr.append(i['pair'])


In [63]:
inr

['B-EDU_USDT',
 'B-VOXEL_USDT',
 'B-KMD_USDT',
 'B-WAVES_USDT',
 'B-COMBO_USDT',
 'B-VIC_USDT',
 'B-XRP_USDT',
 'B-RDNT_USDT',
 'B-JTO_USDT',
 'B-YFI_USDT',
 'B-YGG_USDT',
 'B-RSR_USDT',
 'B-TIA_USDT',
 'B-OP_USDT',
 'B-STEEM_USDT',
 'B-BEAMX_USDT',
 'B-MDT_USDT',
 'B-MEME_USDT',
 'B-ADX_USDT',
 'I-BSW_INR',
 'B-SEI_USDT',
 'I-SPELL_INR',
 'I-FTM_INR',
 'B-JASMY_USDT',
 'B-GAL_USDT',
 'B-ACA_USDT',
 'B-OG_USDT',
 'B-SCRT_USDT',
 'I-DEGO_INR',
 'B-ARK_USDT',
 'B-LUNC_USDT',
 'B-ARKM_USDT',
 'B-CREAM_USDT',
 'B-GMX_USDT',
 'B-GFT_USDT',
 'I-XDC_INR',
 'B-VTHO_USDT',
 'B-BAR_USDT',
 'B-PAXG_USDT',
 'B-VGX_USDT',
 'B-VIDT_USDT',
 'I-CAKE_INR',
 'B-WLD_USDT',
 'B-NEAR_USDT',
 'B-GTC_USDT',
 'B-HIVE_USDT',
 'B-ZIL_USDT',
 'B-KSM_USDT',
 'B-ZRX_USDT',
 'B-HOT_USDT',
 'I-ARKM_INR',
 'I-MKR_INR',
 'B-HOOK_USDT',
 'B-ICP_USDT',
 'B-PEPE_USDT',
 'B-NTRN_USDT',
 'B-MATIC_USDT',
 'B-RLC_USDT',
 'B-WRX_USDT',
 'B-JUV_USDT',
 'B-USTC_USDT',
 'B-SANTOS_USDT',
 'B-DOT_USDT',
 'B-ACM_USDT',
 'B-LQTY_USD